# Collection Metadata Enrichment

Collection metadata can be sourced from multiple backends simultaneously.
The enrichment pipeline composes them at read time via `CollectionMetadataEnricherProtocol`.

## Architecture

```
GET /stac/catalogs/{cat}/collections/{coll}
    │
    ├── PG metadata table (base — always present)
    │
    ├── DriverMetadataEnricher (priority 200)
    │     calls driver.get_collection_metadata() for METADATA-routed drivers
    │     supplements: summaries, providers, item_assets, assets, extent
    │
    └── BigQueryCollectionEnricher (priority 200, GCP only)
          injects real-time row counts / aggregate stats
```

**Enrichers run in ascending `priority` order.**
Each enricher supplements (never overwrites) what previous stages provided.

## `DriverMetadataEnricher` behaviour

When a collection has a METADATA-routed driver (e.g. Iceberg), the enricher:
1. Calls `driver.get_collection_metadata()` — reads table properties or sidecar JSON
2. Merges `summaries`, `providers`, `item_assets`, `assets`, `extra_metadata`, `links`
   into the response — but only for keys not already provided by PG
3. If the driver declares `Capability.AGGREGATION`, also calls `compute_extents()`
   to fill spatial/temporal extent when absent

## Metadata write-through

On every `PATCH /collections/{id}`, updated metadata is propagated to all
WRITE-routed drivers that implement `set_collection_metadata()`.  Failures are
logged per-driver but never block the update.

In [ ]:
import httpx

BASE = "http://localhost:8000"
CATALOG_ID = "demo-enrichment"

client = httpx.AsyncClient(base_url=BASE, timeout=30)
r = await client.post("/stac/catalogs", json={"id": CATALOG_ID, "title": "Enrichment Demo"})
print("Catalog:", r.status_code)

---
## 1. Iceberg as METADATA driver — extents computed at read time

Iceberg stores table-level properties (title, description, keywords, providers)
in its catalogue.  When routed for METADATA, the `DriverMetadataEnricher` reads
these and merges them with PG metadata.  If `Capability.AGGREGATION` is declared,
spatial/temporal extents are computed from snapshot statistics.

In [ ]:
COLL_ICE = "forest-cover-iceberg"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_ICE,
    "title": "Forest Cover (Iceberg)",
    "license": "CC-BY-4.0",
})
print("Collection:", r.status_code)

# Route WRITE and METADATA to Iceberg
routing = {
    "plugin_id": "storage:collections",
    "operations": {
        "WRITE":    [{"driver_id": "iceberg", "on_failure": "fatal"}],
        "READ":     [{"driver_id": "iceberg"}],
        "SEARCH":   [{"driver_id": "elasticsearch"}],
        "METADATA": [{"driver_id": "iceberg"}],
    },
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_ICE}/configs/storage:collections",
    json=routing,
)
print("Routing:", r.status_code)

In [ ]:
import random

# Ingest items — Iceberg driver stores them and computes snapshot stats
features = [
    {
        "type": "Feature",
        "id": f"tile-{i:04d}",
        "geometry": {
            "type": "Polygon",
            "coordinates": [[
                [10 + i * 0.1, 45 + i * 0.1],
                [10 + i * 0.1 + 0.09, 45 + i * 0.1],
                [10 + i * 0.1 + 0.09, 45 + i * 0.1 + 0.09],
                [10 + i * 0.1, 45 + i * 0.1 + 0.09],
                [10 + i * 0.1, 45 + i * 0.1],
            ]],
        },
        "properties": {
            "cover_pct": round(random.uniform(0, 100), 1),
            "year": random.choice([2018, 2020, 2022, 2024]),
        },
    }
    for i in range(15)
]
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_ICE}/items",
    json={"type": "FeatureCollection", "features": features},
)
print("Ingest:", r.status_code)

# Read collection — DriverMetadataEnricher adds Iceberg-computed extents
r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_ICE}")
coll = r.json()
print("\nCollection metadata (enriched):")
print("  title:", coll.get("title"))
print("  extent:", coll.get("extent"))   # filled by DriverMetadataEnricher.compute_extents()
print("  summaries:", coll.get("summaries"))

---
## 2. Metadata write-through

When you update collection metadata via `PATCH`, the change is written to PG
and simultaneously propagated to all WRITE-routed drivers that support
`set_collection_metadata()`.  No extra API call needed.

In [ ]:
# Update PG metadata — write-through propagates to Iceberg table properties
r = await client.patch(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_ICE}",
    json={
        "title": {"en": "Global Forest Cover"},
        "description": {"en": "Global 100m forest cover tiles from Sentinel-2."},
        "keywords": ["forest", "sentinel-2", "land cover", "UNFCCC"],
        "providers": [{"name": "FAO SEPAL", "roles": ["producer"], "url": "https://sepal.io"}],
        "summaries": {"cover_pct": {"minimum": 0, "maximum": 100}},
    },
)
print("PATCH:", r.status_code)

# Read back — both PG and Iceberg metadata are now in sync
r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_ICE}")
coll = r.json()
print("Updated title:", coll.get("title"))
print("Keywords:", coll.get("keywords"))
print("Providers:", [p.get("name") for p in (coll.get("providers") or [])])

---
## 3. DuckDB sidecar JSON metadata

The DuckDB driver stores collection metadata in a sidecar JSON file alongside
the Parquet source (`{dir}/.dynastore_meta_{collection_id}.json`).  The
`DriverMetadataEnricher` reads this at collection read time.

In [ ]:
COLL_DUCK = "crop-yield-parquet"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_DUCK,
    "title": "Crop Yield (DuckDB/Parquet)",
    "license": "CC-BY-4.0",
})
print("Collection:", r.status_code)

# Route READ + METADATA to DuckDB
routing = {
    "plugin_id": "storage:collections",
    "operations": {
        "WRITE":    [{"driver_id": "postgresql"}],
        "READ":     [{"driver_id": "duckdb"}],
        "METADATA": [{"driver_id": "duckdb"}],
    },
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_DUCK}/configs/storage:collections",
    json=routing,
)
print("Routing:", r.status_code)

# Trigger a metadata write-through to create the sidecar JSON
r = await client.patch(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_DUCK}",
    json={
        "summaries": {"crop": ["wheat", "corn", "rice", "soybean"]},
        "providers": [{"name": "GAEZ v4", "roles": ["producer"], "url": "https://gaez.fao.org"}],
    },
)
print("Write-through to DuckDB sidecar:", r.status_code)

# Read back — DuckDB sidecar enriches the response
r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_DUCK}")
coll = r.json()
print("Providers (from DuckDB sidecar):", [p.get("name") for p in (coll.get("providers") or [])])
print("Summaries:", coll.get("summaries"))

---
## 4. Custom enricher — plug in your own source

Implement `CollectionMetadataEnricherProtocol` and register via
`register_plugin()` in your module's `lifespan`.

```python
from dynastore.tools.discovery import register_plugin
from dynastore.models.protocols.enrichment import CollectionMetadataEnricherProtocol

class MyStatsEnricher:
    enricher_id = "my_stats"
    priority = 300   # runs after DriverMetadataEnricher (200)

    def can_enrich(self, catalog_id: str, collection_id: str = "") -> bool:
        return bool(collection_id)   # collection-level only

    async def enrich(
        self, catalog_id, collection_id, metadata, context
    ):
        count = await my_db.count(catalog_id, collection_id)
        return {**metadata, "item_count": count}

# In your module lifespan:
register_plugin(MyStatsEnricher())
```

**Key contract:**
- Never mutate `metadata` in place — return a new dict
- `can_enrich()` must accept optional `collection_id` (also called at catalog level)
- Exceptions are caught per-enricher; failures log a warning but never break reads

In [ ]:
r = await client.delete(f"/stac/catalogs/{CATALOG_ID}?force=true")
print("Cleanup:", r.status_code)
await client.aclose()